<a href="https://colab.research.google.com/github/Aswin-k61/NLP_Repo/blob/main/Student_Performance_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import nltk
import re
import string

from google.colab import files
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
df=pd.read_csv("/content/Student_Performance (1).csv")
df.tail()

,student_id,age,gender,school_type,parent_education,study_hours,attendance_percentage,internet_access,travel_time,extra_activities,study_method,math_score,science_score,english_score,overall_score,final_grade
24995,12047,17,female,public,phd,1.8,55.2,yes,15-30 min,no,mixed,55.8,48.5,46.7,46.1,e
24996,1102,16,female,private,diploma,2.7,97.1,yes,<15 min,no,coaching,64.8,48.2,52.3,56.5,d
24997,4422,19,other,private,post graduate,1.0,63.0,yes,<15 min,no,group study,50.5,20.3,36.1,36.7,f
24998,7858,14,male,private,diploma,1.0,69.4,yes,15-30 min,yes,group study,13.0,34.2,7.3,34.1,f
24999,11621,18,other,public,no formal,0.7,60.3,yes,30-60 min,no,mixed,36.5,45.1,16.5,31.4,f


In [ ]:
df.info()
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   student_id             25000 non-null  int64  
 1   age                    25000 non-null  int64  
 2   gender                 25000 non-null  object 
 3   school_type            25000 non-null  object 
 4   parent_education       25000 non-null  object 
 5   study_hours            25000 non-null  float64
 6   attendance_percentage  25000 non-null  float64
 7   internet_access        25000 non-null  object 
 8   travel_time            25000 non-null  object 
 9   extra_activities       25000 non-null  object 
 10  study_method           25000 non-null  object 
 11  math_score             25000 non-null  float64
 12  science_score          25000 non-null  float64
 13  english_score          25000 non-null  float64
 14  overall_score          25000 non-null  float64
 15  fi

(25000, 16)

In [ ]:
df.isnull().sum()

,0
student_id,0
age,0
gender,0
school_type,0
parent_education,0
study_hours,0
attendance_percentage,0
internet_access,0
travel_time,0
extra_activities,0


In [ ]:
df=df.sample(
    5000,
    random_state=42
)

In [ ]:
df['final_grade']=df['final_grade'].map({
    'a':0,
    'b':1,
    'c':2,
    'd':3,
    'e':4,
    'f':5
}
)

In [ ]:
from nltk.corpus.reader import WordNetCorpusReader
stop_words=set(stopwords.words('english'))
stemmer=PorterStemmer()

def preprocess(text):
  text=text.lower()
  text=re.sub('<.*?>','',text)

 # remove punctuation
  text=text.translate(
  str.maketrans('','',string.punctuation))

 # tokenization
  words=text.split()

 # stopword removal
  words=[
    word for word in words
    if word not in stop_words
  ]

  #stemming
  words=[
      stemmer.stem(word)
      for word in words
   ]


  return " ".join(words)


In [ ]:
vectorizer=TfidfVectorizer(
    max_features=5000,
)

In [ ]:
# Re-initialize X from the original DataFrame and apply preprocessing to ensure it's a DataFrame with preprocessed text columns
X_temp_df = df.drop('final_grade', axis=1)
X_temp_df = X_temp_df.apply(lambda col: col.apply(preprocess) if col.dtype == 'object' else col)

# Identify text columns from the re-initialized DataFrame
text_features = X_temp_df.select_dtypes(include='object')

# Concatenate these text features into a single string per row
# Ensure all values are strings before joining
combined_text = text_features.astype(str).agg(' '.join, axis=1)

# Fit and transform the combined text data
X_vectorized_text = vectorizer.fit_transform(combined_text).toarray()

# Update X to be the vectorized text features (NumPy array)
X = X_vectorized_text

In [ ]:
y=df['final_grade']

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
model=Sequential()
model.add(Dense(128,activation='relu',input_shape=(X_train.shape[1],)))
model.add(Dense(64,activation='relu'))
model.add(Dense(1,activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=32,
)

Epoch 1/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.1082 - loss: -62.3451
Epoch 2/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1085 - loss: -1300.9224
Epoch 3/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1085 - loss: -6482.1582
Epoch 4/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1085 - loss: -18460.1582
Epoch 5/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1085 - loss: -39617.5156
Epoch 6/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1085 - loss: -72159.8281
Epoch 7/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.1085 - loss: -118072.6016
Epoch 8/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.1085 - loss: -178849.9219
Epoch 9/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1085 - loss: -255898.0938
Epoch 10/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.1085 - loss: -350294.6250
Epoch 11/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1085 - loss: -463151.71

In [ ]:
loss,accuracy=model.evaluate(X_test,y_test)
print("Accuracy:",accuracy)

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1080 - loss: -1235630.6250
Accuracy: 0.1080000028014183


### Preparing New Data for Prediction

To predict a new `final_grade`, you'll need to create a new input with similar features to your original dataset, preprocess its text features, and then vectorize them using the `vectorizer` you already fitted.

In [ ]:
# Create a sample new data point (similar to one row of your original DataFrame without 'final_grade')
# Make sure to include all text-based columns that were used in X_temp_df
sample_data = {
    'student_id': [99999],
    'age': [17],
    'gender': ['male'],
    'school_type': ['public'],
    'parent_education': ['graduate'],
    'study_hours': [3.5],
    'attendance_percentage': [85.0],
    'internet_access': ['yes'],
    'travel_time': ['<15 min'],
    'extra_activities': ['yes'],
    'study_method': ['coaching'],
    'math_score': [70.0],
    'science_score': [65.0],
    'english_score': [72.0],
    'overall_score': [69.0]
}

sample_df = pd.DataFrame(sample_data)

# Apply the same preprocessing to the text features of the sample data
sample_df_preprocessed = sample_df.apply(lambda col: col.apply(preprocess) if col.dtype == 'object' else col)

# Extract text features for vectorization
sample_text_features = sample_df_preprocessed.select_dtypes(include='object')

# Combine text features into a single string
sample_combined_text = sample_text_features.astype(str).agg(' '.join, axis=1)

# Use the fitted vectorizer to transform the sample text
sample_X_vectorized = vectorizer.transform(sample_combined_text).toarray()

print("Sample input prepared for prediction:")
print(sample_X_vectorized)


Sample input prepared for prediction:
[[0.43371586 0.         0.         0.         0.50295795 0.
  0.         0.         0.37663701 0.         0.         0.37739389
  0.17911041 0.         0.         0.         0.         0.
  0.         0.30531346 0.         0.         0.         0.
  0.38645235]]


In [ ]:
# Make a prediction using the trained model
prediction = model.predict(sample_X_vectorized)
predicted_class = np.argmax(prediction, axis=1)[0]

grade_map = {
    0: "Grade A",
    1: "Grade B",
    2: "Grade C",
    3: "Grade D",
    4: "Grade E",
    5: "Grade F"
}

print(grade_map.get(predicted_class, "Unknown Grade"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
Grade A
